In [ ]:
import lmdb
import pickle
import io
from PIL import Image
from torch.utils.data import Dataset

class LMDBDataset(Dataset):
    def __init__(self, lmdb_path, transform=None):
        self.lmdb_path = lmdb_path
        self.transform = transform

        # Open once ONLY to read keys, then close
        env = lmdb.open(
            lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            self.keys = [k for k, _ in txn.cursor() if k != b"__len__"]

        env.close()

        #Critical: length derived from keys, not __len__
        self.length = len(self.keys)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= self.length:
            raise IndexError

        # Open LMDB locally (safe for Windows)
        env = lmdb.open(
            self.lmdb_path,
            readonly=True,
            lock=False,
            readahead=False,
            meminit=False,
            subdir=False
        )

        with env.begin() as txn:
            data = pickle.loads(txn.get(self.keys[idx]))

        env.close()

        img = Image.open(io.BytesIO(data["image"])).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, data["label"]

In [ ]:
CLASS_MAP = {
    "Normal": 0,
    "COVID-19": 1,
    "Pneumonia": 2
}

In [ ]:
import lmdb
import pickle
import cv2
import os
import numpy as np

def normalize_image_list(image_names):
    if isinstance(image_names, np.ndarray):
        if image_names.ndim == 0:
            return [image_names.item()]
        return image_names.tolist()
    if isinstance(image_names, str):
        return [image_names]
    return list(image_names)


def build_lmdb_from_folds_and_test(
    folds,
    test_set,
    lmdb_path,
    image_root,
    map_size=1e12
):
    env = lmdb.open(lmdb_path, map_size=int(map_size))

    idx = 0

    with env.begin(write=True) as txn:

        # ---------- K-FOLDS ----------
        for fold_idx in range(5):
            for split_idx in [0, 1]:
                split_name = "train" if split_idx == 0 else "val"
                df = folds[fold_idx][split_idx]

                for _, row in df.iterrows():
                    label = CLASS_MAP[row["finding"]]
                    image_names = normalize_image_list(row["images"])

                    for img_name in image_names:
                        img_path = os.path.join(image_root, img_name)

                        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                        if img is None:
                            raise RuntimeError(f"Could not read {img_path}")

                        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

                        key = f"fold{fold_idx}/{split_name}/{idx:08d}".encode()
                        value = pickle.dumps((img, label))
                        txn.put(key, value)
                        idx += 1

        # ---------- TEST SET ----------
        for _, row in test_set.iterrows():
            label = CLASS_MAP[row["finding"]]
            image_names = normalize_image_list(row["images"])

            for img_name in image_names:
                img_path = os.path.join(image_root, img_name)

                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                if img is None:
                    raise RuntimeError(f"Could not read {img_path}")

                img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

                key = f"test/{idx:08d}".encode()
                value = pickle.dumps((img, label))
                txn.put(key, value)
                idx += 1

    env.sync()
    env.close()

    print(f"LMDB created at: {lmdb_path}")
    print(f"Total samples written: {idx}")

In [ ]:
build_lmdb_from_folds_and_test(
    folds=folds,
    test_set=test_set,
    lmdb_path="COVID-19-dataset.lmdb",
    image_root="./3A_images/"
)